# DistilBERT reference baseline (Phase 4, issue I-8)

The paper's related-work section stops at 2020 with no transformer baseline, positioning
classical ML as if it were still competitive with the state of the art. This notebook adds a
real reference point: `distilbert-base-uncased` fine-tuned on the same label-corrected LIAR
binary task, evaluated with the same metric set (`liar_utils.evaluate_full`) as every other
notebook in this repo, so it is directly comparable to the classical-ML results.

This is a *reference point*, not a competing contribution -- the paper should frame classical
ML as the interpretable, low-resource alternative (I-8), not claim to beat a transformer.

In [1]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label, print_report

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Device: mps


Load data (corrected labels) -- raw `Statement` text, no stemming/stopword removal (the tokenizer handles that)

In [2]:
train_df = load_and_label("train.csv")[["Statement", "Label"]]
valid_df = load_and_label("valid.csv")[["Statement", "Label"]]
test_df = load_and_label("test.csv")[["Statement", "Label"]]

balance = train_df["Label"].value_counts(normalize=True).rename({0: "fake", 1: "real"})
assert 0.35 < balance["fake"] < 0.5, "Fake-class share outside the expected ~44% range -- check label mapping"

train_ds = Dataset.from_pandas(train_df.rename(columns={"Statement": "text", "Label": "labels"}), preserve_index=False)
valid_ds = Dataset.from_pandas(valid_df.rename(columns={"Statement": "text", "Label": "labels"}), preserve_index=False)
test_ds = Dataset.from_pandas(test_df.rename(columns={"Statement": "text", "Label": "labels"}), preserve_index=False)

Tokenize -- LIAR statements are short (single sentences), so max_length=64 covers almost all of them

In [3]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)


train_ds = train_ds.map(tokenize, batched=True)
valid_ds = valid_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

columns = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=columns)
valid_ds.set_format(type="torch", columns=columns)
test_ds.set_format(type="torch", columns=columns)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

Fine-tune for 3 epochs, macro-F1 as the model-selection metric (consistent with every other notebook in this repo)

In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    metrics = evaluate_full(labels, preds)
    return {k: v for k, v in metrics.items() if k != "confusion_matrix"}


model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="/tmp/distilbert_liar_output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    seed=RANDOM_STATE,
    report_to=[],
    metric_for_best_model="macro_f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Fake Precision,Fake Recall,Fake F1,Real Precision,Real Recall,Real F1
1,0.657700,0.665093,0.633178,0.606384,0.717718,0.387987,0.503688,0.603575,0.859281,0.709080
2,0.592200,0.666847,0.634735,0.619491,0.678832,0.452922,0.543330,0.613975,0.802395,0.695652
3,0.495700,0.692381,0.644860,0.639400,0.656863,0.543831,0.595027,0.636951,0.738024,0.683773


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1920, training_loss=0.581846046447754, metrics={'train_runtime': 242.3618, 'train_samples_per_second': 126.753, 'train_steps_per_second': 7.922, 'total_flos': 508674810839040.0, 'train_loss': 0.581846046447754, 'epoch': 3.0})

Final held-out test evaluation -- same metric set as every classical-ML notebook

In [5]:
test_output = trainer.predict(test_ds)
y_test_pred = np.argmax(test_output.predictions, axis=-1)
y_test_true = test_df["Label"].to_numpy()

print_report("DistilBERT (fine-tuned, 3 epochs)", y_test_true, y_test_pred)
test_metrics = evaluate_full(y_test_true, y_test_pred)
test_metrics

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



DistilBERT (fine-tuned, 3 epochs)
[[288 265]
 [179 535]]
              precision    recall  f1-score   support

        fake      0.617     0.521     0.565       553
        real      0.669     0.749     0.707       714

    accuracy                          0.650      1267
   macro avg      0.643     0.635     0.636      1267
weighted avg      0.646     0.650     0.645      1267



{'accuracy': 0.6495659037095501,
 'macro_f1': 0.6357215012821509,
 'fake_precision': 0.6167023554603854,
 'fake_recall': 0.5207956600361664,
 'fake_f1': 0.5647058823529412,
 'real_precision': 0.66875,
 'real_recall': 0.7492997198879552,
 'real_f1': 0.7067371202113606,
 'confusion_matrix': [[288, 265], [179, 535]]}

In [6]:
transformer_results = pd.DataFrame(
    [
        {
            "Pipeline": "Transformer",
            "Method": "Fine-tuned (3 epochs)",
            "Model": "distilbert-base-uncased",
            "Test Accuracy": test_metrics["accuracy"],
            "Test Macro-F1": test_metrics["macro_f1"],
            "Test Fake Precision": test_metrics["fake_precision"],
            "Test Fake Recall": test_metrics["fake_recall"],
            "Test Fake F1": test_metrics["fake_f1"],
            "Test Real F1": test_metrics["real_f1"],
            "Test Confusion Matrix": test_metrics["confusion_matrix"],
        }
    ]
)
transformer_results.to_csv("transformer_baseline_results_v1.csv", index=False)
transformer_results

,Pipeline,Method,Model,Test Accuracy,Test Macro-F1,Test Fake Precision,Test Fake Recall,Test Fake F1,Test Real F1,Test Confusion Matrix
0,Transformer,Fine-tuned (3 epochs),distilbert-base-uncased,0.649566,0.635722,0.616702,0.520796,0.564706,0.706737,"[[288, 265], [179, 535]]"
